## **Predicting Bike Sharing Demand with Regression Models** 
---
By Aditya Nugraha

<!-- ![image: https://www.freepik.com/vectors/ride-bicycle](assets/5243.jpg) -->
<img src="assets/5243.jpg" alt="image: https://www.freepik.com/vectors/ride-bicycle"/>

*Image Source: https://www.freepik.com/vectors/ride-bicycle*

### **Contents**
---

1. Business Problem Understanding
2. Data Understanding
3. EDA & Feature Engineering
4. Modeling
5. Conclusion
6. Recommendation
7. Save Model

### **Business Problem Understanding**
---

**Context**

Sistem penyewaan sepeda adalah sarana penyewaan sepeda dimana proses mendapatkan keanggotaan, sewa, dan pengembalian sepeda secara mandiri melalui jaringan lokasi seperti kios di seluruh kota. Dengan menggunakan sistem ini, orang dapat menyewa sepeda dari satu lokasi dan mengembalikannya ke lokasi lain sesuai kebutuhan. Menurut data yang di rilis pada situs [Statista.com](https://www.statista.com/chart/14542/bike-sharing-programs-worldwide/) Hingga pertengahan tahun 2018, ada sekitar 1600 program penyewaan sepeda dengan 18,2 juta sepeda di lebih dari 1000 kota.

<img src="assets/14542.jpeg" alt="image: https://www.statista.com/chart/14542/bike-sharing-programs-worldwide/" width="600"/>

*Image Source: https://www.statista.com/chart/14542/bike-sharing-programs-worldwide/*

**Problem Statement**

Terlepas dari efek positifnya pada lalu lintas, masalah lingkungan dan kesehatan, sistem penyewaan sepeda sangat menarik karena data yang mereka hasilkan dapat digunakan untuk tujuan penelitian. Waktu keberangkatan dan kedatangan, lokasi dan durasi penggunaan dicatat dalam sistem ini. Data tersebut dapat digunakan untuk memprediksi kapan waktu, kondisi, dan tempat yang tepat bagi masyarakat untuk melakukan penyewaan sepeda, sekaligus menjadi masalah tersendiri bagi penyedia sistem penyewaan sepeda ini untuk **mengetahui seberapa banyak sepeda yang harus di distribusikan agar sepeda mereka dapat digunakan secara efisien sehingga mengurangi kemungkinan adanya kondisi yang mempunyai banyak sepeda namun sedikit yang menggunakannya ataupun sebaliknya.**

**Goals**

Berdasarkan permasalahan yang telah dijelaskan sebelumnya, pihak yang mengadakan program penyewaan sepeda perlu memiliki 'tool' yang dapat digunakan untuk **memprediksi seberapa banyak sepeda yang harus di distribusikan berdasarkan kondisi tertentu**. adanya perbedaan kondisi dan waktu dapat menambah keakuratan dalam memprediksi banyaknya sepeda yang digunakan, yang mana dapat mendistribusikan sepeda secara efisien sehingga penyedia dapat memaksimalkan keuntungan dari penyewaan tersebut.

**Analytics Approach**

Jadi, yang perlu dilakukan adalah menganalisis data untuk menemukan pola dari fitur-fitur yang ada, yang membedakan pada kondisi dan waktu tertentu.

Selanjutnya, kita akan membangun suatu model untuk memprediksi suatu nilai numerik atau biasa disebut dengan model regresi, model ini dapat digunakan sebagai 'tool' yang membantu penyedia sistem penyewaan sepeda ini untuk memprediksi banyaknya penyewa berdasarkan kondisi dan waktu tertentu.

**Metrics Evaluation**

Evaluasi metrik yang kita gunakan yang paling difokuskan adalah RMSE dan RMSLE, namun kita juga akan menggunakan MAE dan MAPE sebagai pembanding pada kasus ini.
* RMSE merupakan nilai rataan akar kuadrat dari error, metrik ini dapat digunakan untuk mengukur seberapa jauh nilai prediksi yang dihasilkan berdasarkan rataannya.
* RMSLE adalah nilai rataan akar kuadrat dari error yang digunakan khusus jika fitur target ditransformasikan menggunakan perhitungan logaritma, kita dapat mengunakan metrik ini untuk menghitung rasio antara nilai aktual dan prediksinya. nilai dari metrik ini dapat mereduksi error yang ditimbulkan oleh outliers atau perbedaan nilai yang terlalu jauh.
* MAE adalah rataan nilai absolut dari error
* MAPE adalah rataan persentase error yang dihasilkan oleh model regresi, dengan nilai keakuratan berdasarkan buku dari [Lewis's (1982)](https://www.nrpa.org/globalassets/journals/jlr/2003/volume-35/jlr-volume-35-number-4-pp-441-454.pdf) mengatakan bahwa jika nilai MAPE sebesar 0.10 atau 10% merupakan prediksi yang sangat akurat, 11-20% merupakan prediksi yang baik, 21-50% merupakan prediksi yang masuk akal, dan 51% dan lebih banyak lagi adalah prediksi yang tidak akurat, adapun jika lebih dari 100% merupakan prediksi yang sangat tidak akurat.

Semakin kecil nilai RMSLE, MAE, dan MAPE yang dihasilkan, berarti model semakin akurat dalam memprediksi banyaknya penyewa sepeda dengan limitasi fitur yang digunakan. 

### **Importing Libraries**
---

In [ ]:
from sklearn.model_selection import cross_val_score, TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_squared_error ,mean_squared_log_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
import category_encoders as ce
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost.sklearn import XGBRegressor
from lightgbm import LGBMRegressor
import pandas as pd
import pickle
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')


def inverse_log1p_nonnegative(values):
    return np.maximum(np.expm1(values), 0)


**Library Version Used**

In [ ]:
import sklearn
import lightgbm
import xgboost
import matplotlib
from platform import python_version

lib = {'scikit-learn': sklearn, 'category_encoder': ce, 'xgboost': xgboost,
       'lightgbm': lightgbm, 'pandas': pd, 'numpy': np, 'seaborn': sns,
       'matplotlib': matplotlib}

for i in lib:
    print(lib[i].__version__, i)
else:
    print(pickle.format_version, 'pickle')
    print(python_version(), 'python')


### **Data Understanding**
---

Sistem penyewaan sepeda sangat berkorelasi dengan kondisi lingkungan dan musim. Misalnya, kondisi cuaca,
curah hujan, hari dalam seminggu, musim, jam dalam sehari, dll. hal tersebut dapat mempengaruhi perilaku penyewa apakah seseorang dengan kondisi tersebut ingin menyewa sepeda atau tidak.

* Dataset menunjukkan data sewa sepeda per jam selama dua tahun (2011 dan 2012). 
* Memiliki 11 atribut yang masing-masing menunjukan waktu penyewaan, kondisi lingkungan, dan banyaknya sepeda yang disewa

|Atribute|Data Type|Description|
|---|---|---|
|dteday|Datetime|Date|
|hum|Float|Normalized Humadity. The values are divided to 100 (maximum value)|
|weathersit|Integer|1: Clear, few clouds, partly cloudy<br>2: Mist, cloudy, mist and broken clouds, mist and few clouds<br>3: Light snow, light rain and thunderstorm and scattered clouds, light rain and scattered clouds<br>4: Heavy rain, ice pallets, thunderstorm and mist, snow and fog|
|holiday|Integer|Whether the day is holiday or not (1: holiday, 0: not holiday). References: [2011](https://dchr.dc.gov/page/holiday-schedules-2010-and-2011), [2012](https://dchr.dc.gov/page/holiday-schedules-2012-and-2013)|
|season|Integer|Season (1: Winter, 2: Spring, 3: Summer, 4: Fall)|
|atemp|Float|Normalized feeling temperature in Celsius. The values are divided to 50 (maximum value)|
|temp|Float|Normalized temperature in Celsius. The values are divided to 41 (maximum value)|
|hr|Integer|Hour (0 to 23)|
|casual|Integer|Count of casual users|
|registered|Integer|Count of registered users|
|cnt|Integer|Count of hourly total rental bikes including both casual and registered (Target variable)|

In [ ]:
df=pd.read_csv('data_bike_sharing.csv', parse_dates=['dteday'])


In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.dtypes

**Chronological Train/Test Split**


In [ ]:
# The chronological split is created in the modeling section after feature cleaning.


### **EDA & Feature Engineering**
---

**Null and Duplicated Values**

In [ ]:
df.isna().any()

In [ ]:
df.duplicated().any()

Berdasarkan hasil diatas, tidak ditemukan adanya null values maupun data yang terduplikasi

**Drop Unused Features**

Kita akan menghilangkan fitur yang dirasa tidak perlu digunakan seperti  `casual` dan `registered` yang akan di total dan menjadi fitur target yaitu `cnt`, jika di pertahankan maka dikhawatirkan akan membuat model menjadi bias atau tidak sesuai.

In [ ]:
df.drop(['casual', 'registered'], axis=1, errors='ignore', inplace=True)
df.columns

**Retaining Real Demand Peaks**


In [ ]:
bp = df.drop('dteday', axis=1).columns.tolist()
n = 1

plt.figure(figsize=(12,7))
for i in bp:
    plt.subplot(3,3,n)
    df.boxplot(i)
    n += 1

The count target contains genuine demand peaks. Because the dataset does not identify these values as data errors, we retain them. We also avoid computing a filtering threshold from the full time series before splitting, which would use information from the future test period. The target is log-transformed inside each model during fitting instead.


In [ ]:
# Retain every observation; high counts may be genuine demand peaks.
print(f'Retained all {len(df):,} observations.')


**Handling Skewness with Log Transformation**

In [ ]:
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.boxplot(df['cnt'])
plt.title('Before Transformation')

plt.subplot(2, 2, 2)
plt.boxplot(df['cnt'].apply(lambda x: np.log1p(x)))
plt.title('After Transformation')

plt.subplot(2, 2, 3)
df['cnt'].hist()

plt.subplot(2, 2, 4)
df['cnt'].apply(lambda x: np.log1p(x)).hist()

plt.show()


We use `log1p` on the target inside each `TransformedTargetRegressor` to reduce the influence of large count differences while fitting. Predictions are converted back to bike-count units before evaluation. This keeps every observed row and avoids estimating outlier cutoffs using future data.


In [ ]:
original_skew = df['cnt'].skew()
cnt_log = np.log1p(df['cnt'])

print('Before log1p transformation:', original_skew)
print('After log1p transformation:', cnt_log.skew())


The exact skew values are calculated when the notebook runs. The modeling pipeline applies `log1p` to `cnt` during fitting and converts predictions back to the original count scale for evaluation.


**Categorical Features**

In [ ]:
time_attr = ['weathersit', 'holiday', 'season', 'hr']

for i in time_attr:
    l = len(df[i].unique())
    if l == 2:
        aspect = .6
    elif l == 4:
        aspect = 1
    elif l == 7:
        aspect = 1.6
    elif l == 12:
        aspect = 2.1  
    elif l == 24:
        aspect = 3.1
    elif l == 31:
        aspect = 4.1
        
    sns.catplot(x=i, y='cnt', data=df,
                kind='bar', height=5, aspect=aspect)
    plt.title(i, size=15, weight='bold')
    plt.xlabel(None)
    plt.show()


**Numerical Features**

In [ ]:
num_attr = ['hum', 'atemp', 'temp']

for i in num_attr:
    sns.displot(df[i], kde=True, stat='density', aspect=2)
    plt.title(i, size=15, weight='bold')
    plt.xlabel(None)
    plt.show()

sns.displot(cnt_log, kde=True, stat='density', aspect=2)
plt.title('cnt after log1p transformation', size=15, weight='bold')
plt.xlabel(None)
plt.show()


### **Modeling**
---

**Custom Transformer to Extract `dteday` Features**

The `dteday` feature is expanded into year, month, day, weekday, and working-day features. In pandas, weekday is numbered Monday = 0 through Sunday = 6.

|Feature|Description|
|---|---|
|year|Year (2011 to 2012)|
|month|Month (1 to 12)|
|day|Day (1 to 31)|
|weekday|Day of the week (0: Monday, ..., 6: Sunday)|
|workingday|1 when the date is not a weekend or holiday; otherwise 0|


In [ ]:
class DateTransformer(BaseEstimator, TransformerMixin):  
    
    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        X = X.copy()
        X['year'] = X['dteday'].dt.year
        X['month'] = X['dteday'].dt.month
        X['day'] = X['dteday'].dt.day
        X['weekday'] = X['dteday'].dt.day_of_week
        X['workingday'] = np.where((X['holiday'] == 1) | (
            X['weekday'] == 5) | (X['weekday'] == 6), 0, 1)
        X = X.drop('dteday', axis=1)
        # Only cast the categorical columns to str so that
        # hum, temp, atemp remain float for StandardScaler.
        cat_cols = ['season', 'year', 'weekday', 'weathersit',
                    'holiday', 'month', 'day', 'hr', 'workingday']
        X[cat_cols] = X[cat_cols].astype(str)
        return X

**Chronological Data Split**

Rows are sorted by `dteday` and `hr`. The earliest 80% are used for training and time-ordered cross-validation; the latest 20% are reserved for final evaluation. `TimeSeriesSplit` preserves order within the training period.


In [ ]:
# dteday is parsed as a datetime when the CSV is loaded (cell 15).
df = df.sort_values(['dteday', 'hr']).reset_index(drop=True)

# casual and registered are components of cnt and must not enter X.
assert not {'casual', 'registered'}.intersection(df.columns), \
    'Drop casual and registered before building the feature matrix.'

X = df.drop(columns=['cnt'])
y = df['cnt'].astype(float)

split_idx = int(len(df) * 0.8)
X_train = X.iloc[:split_idx].copy()
X_test = X.iloc[split_idx:].copy()
y_train = y.iloc[:split_idx].copy()
y_test = y.iloc[split_idx:].copy()


**Encoding** 

kita perlu melakukan encoding pada fitur kategorik karena komputer tidak dapat memproses data berbentuk kategorik, walaupun nilainya sudah berupa angka namun itu adalah nilai kategorik, jika di proses dengan data seperti itu maka akan membuat model salah dalam melakukan prediksi.

**One-Hot:** season, year, weekday, weathersit, holiday  
**Binary:** month, day, hr

In [ ]:
transformer = ColumnTransformer([
    ('One Hot', OneHotEncoder(drop='first', handle_unknown='ignore'),
     ['season', 'year', 'weekday', 'weathersit', 'holiday']),
    ('Binary', ce.BinaryEncoder(), ['month', 'day', 'hr'])
], remainder='passthrough')


**Simple Model Benchmarking**

Pada tahap ini kita akan melakukan proses pencarian skor dari metrik yang telah kita tentukan sebelumnya, algoritma yang digunakan antara lain `LinearRegression`, `Decision Tree`, `Random Forest`, `XGBoost`, dan `LightGBM`. kita juga menggunakan `Standard Scaler` karena merupakan yang paling tepat untuk kasus regresi dengan tidak terlalu banyak outlier. 

In [ ]:
lr = LinearRegression()
dt = DecisionTreeRegressor(random_state=42)
rf = RandomForestRegressor(random_state=42)
xgb = XGBRegressor(random_state=42)
lgb = LGBMRegressor(random_state=42)

# Fit each estimator to log1p(cnt), then return predictions in bike-count units.
models = [lr, dt, rf, xgb, lgb]

scaler = StandardScaler()

score_rmse = []
nilai_mean_rmse = []
nilai_std_rmse = []

score_msle = []
nilai_mean_msle = []
nilai_std_msle = []

score_mae = []
nilai_mean_mae = []
nilai_std_mae = []

score_mape = []
nilai_mean_mape = []
nilai_std_mape = []

for regressor in models:
    crossval = TimeSeriesSplit(n_splits=5)
    estimator = Pipeline([
        ('dates', DateTransformer()),
        ('preprocessing', transformer),
        ('scaler', scaler),
        ('model', TransformedTargetRegressor(
            regressor=regressor,
            func=np.log1p,
            inverse_func=inverse_log1p_nonnegative
        ))
    ])

    model_cv_rmse = cross_val_score(estimator, X_train, y_train, cv=crossval,
                                    scoring='neg_root_mean_squared_error', error_score='raise')
    score_rmse.append(model_cv_rmse)
    nilai_mean_rmse.append(model_cv_rmse.mean())
    nilai_std_rmse.append(model_cv_rmse.std())

    model_cv_msle = cross_val_score(estimator, X_train, y_train, cv=crossval,
                                    scoring='neg_mean_squared_log_error', error_score='raise')
    score_msle.append(model_cv_msle)
    nilai_mean_msle.append(model_cv_msle.mean())
    nilai_std_msle.append(model_cv_msle.std())

    model_cv_mae = cross_val_score(estimator, X_train, y_train, cv=crossval,
                                   scoring='neg_mean_absolute_error', error_score='raise')
    score_mae.append(model_cv_mae)
    nilai_mean_mae.append(model_cv_mae.mean())
    nilai_std_mae.append(model_cv_mae.std())

    model_cv_mape = cross_val_score(estimator, X_train, y_train, cv=crossval,
                                    scoring='neg_mean_absolute_percentage_error', error_score='raise')
    score_mape.append(model_cv_mape)
    nilai_mean_mape.append(model_cv_mape.mean())
    nilai_std_mape.append(model_cv_mape.std())


In [ ]:
# Evaluasi hasil dari 5 kandidat algoritma yang digunakan
pd.DataFrame({
    'Model': ['Linear Regressor', 'DecisionTree Regressor', 'RandomForest Regressor', 'XGBoost Regressor', 'LightGBM Regressor'],
    'Mean_RMSE': nilai_mean_rmse,
    'Std_RMSE': nilai_std_rmse,
    'Mean_MSLE': nilai_mean_msle,
    'Std_MSLE': nilai_std_msle,
    'Mean_MAE': nilai_mean_mae,
    'Std_MAE': nilai_std_mae,
    'Mean_MAPE': nilai_mean_mape,
    'Std_MAPE': nilai_std_mape
})

After rerunning this cell, compare the five candidates using training-only `TimeSeriesSplit` results. Use those cross-validation results to decide whether XGBoost is a suitable tuning candidate. Do not select a model based on the held-out final 20%. Scikit-learn reports these losses as negative scores, so values closer to zero indicate lower error.


In [ ]:
from sklearn import metrics

metrics.get_scorer_names()


**Untuned XGBoost Baseline (Held-Out Final Period)**


In [ ]:
baseline_xgb = Pipeline([
    ('dates', DateTransformer()),
    ('preprocessing', transformer),
    ('scaler', scaler),
    ('model', TransformedTargetRegressor(
        regressor=xgb,
        func=np.log1p,
        inverse_func=inverse_log1p_nonnegative
    ))
])

baseline_xgb.fit(X_train, y_train)
y_pred = baseline_xgb.predict(X_test)

score_before_tuning = pd.DataFrame({
    'RMSE': [np.sqrt(mean_squared_error(y_test, y_pred))],
    'RMSLE': [np.sqrt(mean_squared_log_error(y_test, y_pred))],
    'MAE': [mean_absolute_error(y_test, y_pred)],
    'MAPE': [mean_absolute_percentage_error(y_test, y_pred)]
}, index=['XGB'])
score_before_tuning


This is the untuned XGBoost baseline on the final chronological holdout. All metrics are computed in bike-count units (RMSLE applies its logarithm internally). The final period is for reporting only; model choice and hyperparameter selection must use training-period cross-validation.


**Hyperparameter Tuning**

This notebook tunes XGBoost using `RandomizedSearchCV` and five-fold `TimeSeriesSplit`. The search runs only on the training period. It refits the candidate with the lowest mean squared logarithmic error (MSLE); the held-out final period remains untouched until final evaluation.

See the [XGBoost parameter documentation](https://xgboost.readthedocs.io/en/stable/parameter.html) for the search settings.


In [ ]:
# Tree depth
max_depth = list(np.arange(1, 16))

# Learning rate
learning_rate = list(np.arange(1, 100) / 100)

# Number of trees
n_estimators = list(np.arange(100, 201))

# Fraction of training rows used for each tree
subsample = list(np.arange(2, 10) / 10)

# Minimum loss reduction for a further partition
gamma = list(np.arange(1, 11))

# Fraction of columns sampled for each tree
colsample_bytree = list(np.arange(1, 10) / 10)

# L1 regularization
reg_alpha = list(np.logspace(-3, 1, 10))

hyperparam_space = {
    'model__regressor__max_depth': max_depth,
    'model__regressor__learning_rate': learning_rate,
    'model__regressor__n_estimators': n_estimators,
    'model__regressor__subsample': subsample,
    'model__regressor__gamma': gamma,
    'model__regressor__colsample_bytree': colsample_bytree,
    'model__regressor__reg_alpha': reg_alpha
}


In [ ]:
xgb = XGBRegressor(random_state=42, verbosity=0)

estimator_xgb = Pipeline([
    ('dates', DateTransformer()),
    ('preprocessing', transformer),
    ('scaler', scaler),
    ('model', TransformedTargetRegressor(
        regressor=xgb,
        func=np.log1p,
        inverse_func=inverse_log1p_nonnegative
    ))
])

crossval = TimeSeriesSplit(n_splits=5)

random_search = RandomizedSearchCV(
    estimator_xgb,
    param_distributions=hyperparam_space,
    n_iter=50,
    cv=crossval,
    scoring=['neg_root_mean_squared_error', 'neg_mean_squared_log_error',
             'neg_mean_absolute_error', 'neg_mean_absolute_percentage_error'],
    n_jobs=-1,
    refit='neg_mean_squared_log_error',
    random_state=42,
    verbose=1
)


In [ ]:
random_search.fit(X_train, y_train)

In [ ]:
# Rank candidates by training-period MSLE, then inspect RMSE, MAE, and MAPE.
pd.DataFrame(random_search.cv_results_).sort_values(
    by=['rank_test_neg_mean_squared_log_error',
        'rank_test_neg_mean_absolute_error',
        'rank_test_neg_mean_absolute_percentage_error']
).head()


In [ ]:
# Cek skor dan params terbaik
print('XGBoost')
print('Best_score:', random_search.best_score_)
print('Best_params:', random_search.best_params_)

**Tuned XGBoost: Final Evaluation**

After the search completes, fit the selected estimator on the full training period and evaluate it once on the held-out final period.


In [ ]:
xgb_tuning = random_search.best_estimator_
xgb_tuning.fit(X_train, y_train)

y_pred_xgb_tuning = xgb_tuning.predict(X_test)

rmse_xgb_tuning = np.sqrt(mean_squared_error(y_test, y_pred_xgb_tuning))
rmsle_xgb_tuning = np.sqrt(mean_squared_log_error(y_test, y_pred_xgb_tuning))
mae_xgb_tuning = mean_absolute_error(y_test, y_pred_xgb_tuning)
mape_xgb_tuning = mean_absolute_percentage_error(y_test, y_pred_xgb_tuning)


**Performance Comparison**

In [ ]:
final_score = {'XGB Before Tuning': {'RMSE': score_before_tuning.loc['XGB']['RMSE'],
                              'RMSLE': score_before_tuning.loc['XGB']['RMSLE'],
                              'MAE': score_before_tuning.loc['XGB']['MAE'],
                              'MAPE': score_before_tuning.loc['XGB']['MAPE']},
               'XGB After Tuning': {'RMSE': rmse_xgb_tuning, 'RMSLE': rmsle_xgb_tuning, 
                                    'MAE': mae_xgb_tuning, 'MAPE': mape_xgb_tuning},}
pd.DataFrame(final_score).T

Review the rerun results before concluding whether tuning improved the model. Lower values indicate lower error; the selected search objective is MSLE. Do not infer an improvement until the fresh chronological results are available.


In [ ]:
plt.figure(figsize=(12, 7))
sns.scatterplot(x=y_test, y=y_pred).set(
    title='Actual vs. Predicted Bike Count Before Tuning',
    xlabel='Actual Bike Count', ylabel='Predicted Bike Count'
)
plt.show()

plt.figure(figsize=(12, 7))
sns.scatterplot(x=y_test, y=y_pred_xgb_tuning).set(
    title='Actual vs. Predicted Bike Count After Tuning',
    xlabel='Actual Bike Count', ylabel='Predicted Bike Count'
)
plt.show()


Use the rerun plots to inspect prediction errors across low- and high-demand periods. Any conclusion about where errors increase should be based on these chronological holdout results.


**Feature Importances**

Untuk dapat mengetahui fitur apa saja yang paling memengaruhi target, kita dapat mengeceknya menggunakan function feature_importances_.

In [ ]:
estimator_xgb = Pipeline([
    ('dates', DateTransformer()),
    ('preprocessing', transformer),
    ('scaler', scaler),
    ('model', TransformedTargetRegressor(
        regressor=XGBRegressor(random_state=42, verbosity=0),
        func=np.log1p,
        inverse_func=inverse_log1p_nonnegative
    ))
])

estimator_xgb.fit(X_train, y_train)


In [ ]:
feature_names = transformer.get_feature_names_out()
feature_imp_before = pd.Series(
    estimator_xgb['model'].regressor_.feature_importances_, index=feature_names
).sort_values(ascending=False).head()
feature_imp_after = pd.Series(
    xgb_tuning['model'].regressor_.feature_importances_, index=feature_names
).sort_values(ascending=False).head()

feature_imp_before.plot(kind='barh', title='Feature Importances Before Tuning')
plt.show()
feature_imp_after.plot(kind='barh', title='Feature Importances After Tuning')
plt.show()


**Predicted vs Actual Values**

Compare the untuned and tuned predictions with actual demand on the held-out chronological period.


In [ ]:
validation = pd.DataFrame({
    'prediction_before_tuning': y_pred,
    'prediction_after_tuning': y_pred_xgb_tuning,
    'actual': y_test
})
validation


### **Conclusion**
---

The notebook compares an untuned XGBoost baseline with a tuned model using a chronological holdout. Fill in the conclusion only after reviewing the newly computed RMSE, RMSLE, MAE, MAPE, residual plots, and feature importances. The previous shuffled-split scores and conclusions are withdrawn.


### **Recommendation**
---

Untuk mengembangkan model agar dapat menjadi lebih baik lagi dapat dilakukan hal-hal berikut ini:
1. Mengecek performa pada model lain yang tidak kita gunakan saat ini seperti Ridge, Lasso, ataupun yang lebih kompleks seperti Neural Network, dll. hal ini dapat meningkatkan kemungkinan adanya model yang berpotensi menjadi lebih baik lagi. terutama pada model LightGBM yang tidak jadi dipakai karena masalah kompatibilitas, bisa dibuatkan fungsi untuk pengecekan performa model agar dapat mengeluarkan metrik dari RMSLE yang telah kita jadikan acuan utama pada penelitian ini.

1. Lakukan penambahan fitur yang mungkin akan mempunyai korelasi tinggi terhadap target, misalkan seperti lokasi, kecepatan angin, perkiraan cuaca beberapa waktu kedepan, lama sepeda digunakan, dll. jika memungkinkan, lakukan penambahan data secara signifikan dengan mungkin memasukan data dari wilayan lain yang tidak tercakup pada data yang kita gunakan, hal ini perlu dilakukan untuk menambah keakuratan model yang dihasilkan.

1. Optimalkan parameter yang digunakan pada hyperparameter tuning, seperti yang telah diteliti sebelumnya bahwa hyperparameter tuning yang kita lakukan malah membuat performa model menurun, hal ini terjadi karena adanya limitasi pada pengetahuan yang dimiliki sehingga parameter yang digunakan hanya berdasarkan 'trial and error', diharapkan kedepannya dapat meningkatkan pengetahuannya pada hal tersebut.

1. Dataset yang digunakan memungkinkan kita untuk melakukan 'anomaly detection', yaitu melakukan pendeteksian jika terjadi hal diluar kebiasaan, misal seperti jam 1 malam yang seharusnya hanya sedikit atau mungkin tidak ada sama sekali yang menyewa sepeda namun yang terjadi saat itu adalah sebaliknya, sepeda yang disewa pada saat itu malah sangat tinggi yang mungkin disebabkan oleh faktor eksternal seperti misalnya adanya konser ataupun acara besar lainnya didekat lokasi penyewaan sepeda pada waktu tersebut, hal ini membuat sistem mendeteksi adanya kebiasaan yang tidak wajar, fungsinya sendiri karena penyewaan sepeda ini dilakukan secara online dan terintegrasi maka penyedia penyewaan sepeda dapat secara cepat melakukan distribusi pada area tersebut jika memungkinkan.

### **Save, Load, Predict Model with Pickle**
---

**Save Model**

The saved model is the XGBoost estimator selected by five-fold `TimeSeriesSplit` on the training period. Evaluate its final-period scores before describing its expected forecasting performance.


In [ ]:
filename = 'Model_BikeSharing_XGB.sav'

# Save the tuned pipeline after rerunning the corrected notebook.
pickle.dump(xgb_tuning, open(filename, 'wb'))


**Load Model**

In [ ]:
# Don't forget to run libraries section
loaded_model = pickle.load(open(filename, 'rb'))

**Predict New Data**

In [ ]:
# Read New Dataset
# test = pd.read_csv()

In [ ]:
X_new = test.drop(['cnt'], axis=1)
y_new = test['cnt']


In [ ]:
y_pred_test = loaded_model.predict(X_new)

def new_model(scoring):
    return scoring(y_new, y_pred_test)


In [ ]:
print('RMSLE:', np.sqrt(new_model(mean_squared_log_error)))
print('MAE:', new_model(mean_absolute_error))
print('MAPE:', new_model(mean_absolute_percentage_error))

In [ ]:
prediction = pd.DataFrame({'prediction': y_pred_test, 'actual': y_new})
prediction['prediction'] = prediction['prediction'].clip(lower=0)
# prediction.to_csv('prediction.csv', index=False)
prediction
